<a href="https://colab.research.google.com/github/sankeawthong/Project-1-Lita-Chatbot/blob/main/%5B20260328%5D%20IoMT%20LR-MLP%20%E2%80%94%20Full%20revision%20with%20W3%20-W4%20adversarial%20diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [20260322] IoMT LR-MLP — Full revision with W3 adversarial diagnostics
**Adds to original notebook:**
- Working FGSM and PGD (sklearn-based, LR gradient approximation)
- Q1 diagnostic: confusion matrix under fixed clean threshold (FN count under attack)
- Q2 diagnostic: score distribution shift under FGSM/PGD
- τ₉₅ ceiling-effect demonstration
- 95% CI overlap test for CIC-IoMT FPR improvement
- Seed-averaged adversarial summary table (reproduces Table 6)
All original functions preserved unchanged.

In [11]:
import os, sys, json, time, math, glob, argparse
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, log_loss, brier_score_loss,
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm

print("[OK] Imports complete")

[OK] Imports complete


In [12]:
# ----------------------
# Global configuration
# ----------------------
CFG = {
    "paths": {
        "nf":        "/content/Dataset_NF-ToN-IoT.csv",
        "cic_train": "/content/CIC_IoMT_2024_WiFi_MQTT_train.csv",
        "cic_test":  "/content/CIC_IoMT_2024_WiFi_MQTT_test.csv",
        "outdir":    "./outputs",
    },
    "columns": {
        "nf_label_binary":  "Label",
        "nf_label_mc":      "Class",
        "cic_label_binary": "label",
        "cic_label_mc":     "Class",
    },
    "train": {
        "test_size":               0.2,
        "random_state":            42,
        "use_smote":               True,
        "smote_kind":              "smote_tomek",
        "mlp_hidden_units":        64,
        "mlp_dropout":             0.2,
        "batch_size":              2048,
        "max_epochs":              25,
        "early_stopping_patience": 5,
    },
    "adv": {
        "eps_list":       [0.01, 0.02, 0.05, 0.10],
        "pgd_steps":      10,
        "pgd_alpha_factor": 0.02,
    },
    "calibration": {
        "method":    "temperature",
        "n_bins_ece": 15,
        "calib_frac": 0.1,
    },
    "tiny_slice": {
        "enabled":     True,
        "benign_frac": 0.015,
        "seed":        42,
    },
}
SEEDS = [42, 43, 44, 45, 46]
print("[OK] CFG loaded")

[OK] CFG loaded


In [13]:
# ================================
# Utility + metric helpers
# ================================
def ensure_outdir(path):
    out = Path(path)
    out.mkdir(parents=True, exist_ok=True)
    return str(out)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def load_csv(path):
    df = pd.read_csv(path, low_memory=False)
    # Drop unnamed index columns
    df = df.loc[:, ~df.columns.str.match(r'^Unnamed')]
    return df

def build_feature_list(df, drop_cols):
    return [c for c in df.columns if c not in drop_cols]

def build_binary_labels(df, bin_col, pos_label=None):
    y = df[bin_col].values
    if y.dtype == object:
        unique_labels = np.unique(y)
        if 'Benign' in unique_labels or 'Benign_test' in unique_labels:
            return np.where((y == 'Benign') | (y == 'Benign_test'), 0, 1)
        else:
            return np.ones(len(y), dtype=int)
    return y.astype(int)

def fpr_at_dr(y_true, scores, target_dr=0.95):
    y_true  = np.asarray(y_true).astype(int)
    scores  = np.asarray(scores)
    pos_scores = scores[y_true == 1]
    if len(pos_scores) == 0:
        return np.nan, np.nan
    thr   = np.quantile(pos_scores, 1.0 - target_dr)
    y_hat = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn + 1e-12)
    return fpr, thr

def expected_calibration_error(y_true, prob_pos, n_bins=15):
    probas = np.column_stack([1 - prob_pos, prob_pos])
    confidences = probas.max(axis=1)
    predictions = probas.argmax(axis=1)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        in_bin = (confidences > bins[i]) & (confidences <= bins[i+1])
        if in_bin.any():
            acc  = (y_true[in_bin] == predictions[in_bin]).mean()
            conf = confidences[in_bin].mean()
            ece += np.abs(acc - conf) * in_bin.mean()
    return ece

print("[OK] Helpers loaded")

[OK] Helpers loaded


In [14]:
# ================================
# LR-MLP training (unchanged)
# ================================
def make_smote(kind="smote_tomek", random_state=42):
    if kind == "smote":      return SMOTE(random_state=random_state)
    elif kind == "smote_tomek": return SMOTETomek(random_state=random_state)
    return None

def fit_lr_then_mlp(X_train, y_train, X_eval, y_eval,
                    use_smote=True, seed=42, max_epochs=25,
                    batch_size=2048, hidden_units=64):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_eval_s  = scaler.transform(X_eval)

    if use_smote and len(np.unique(y_train)) > 1:
        sampler  = make_smote(CFG["train"]["smote_kind"], random_state=seed)
        X_train_s, y_train = sampler.fit_resample(X_train_s, y_train)

    lr = LogisticRegression(penalty="l2", C=1.0, solver="lbfgs",
                            max_iter=1000, class_weight="balanced",
                            n_jobs=-1, random_state=seed)

    synthetic_sample_added = False
    if len(np.unique(y_train)) < 2:
        print("[WARN] Single class — adding synthetic benign sample.")
        synthetic_x = X_train_s.mean(axis=0).reshape(1, -1)
        X_train_s   = np.vstack([X_train_s, synthetic_x])
        y_train     = np.append(y_train, 0)
        synthetic_sample_added = True

    lr.fit(X_train_s, y_train)
    z_train = lr.decision_function(X_train_s).reshape(-1, 1)
    z_eval  = lr.decision_function(X_eval_s ).reshape(-1, 1)

    mlp = MLPClassifier(
        hidden_layer_sizes=(hidden_units,), activation="relu",
        solver="adam", alpha=0.0,
        batch_size=batch_size, learning_rate_init=1e-3,
        max_iter=max_epochs,
        early_stopping=not synthetic_sample_added,
        n_iter_no_change=CFG["train"]["early_stopping_patience"],
        validation_fraction=0.1 if not synthetic_sample_added else 0.0,
        random_state=seed,
    )
    mlp.fit(z_train, y_train)

    class Pipe:
        def __init__(self, sc, lr_):
            self.scaler = sc
            self.lr     = lr_
        def transform(self, X):
            return self.lr.decision_function(self.scaler.transform(X)).reshape(-1, 1)

    return Pipe(scaler, lr), mlp

print("[OK] fit_lr_then_mlp loaded")

[OK] fit_lr_then_mlp loaded


In [15]:
# ============================================================
# Adversarial generation — FGSM + PGD (sklearn LR-gradient)
# ============================================================
# The LR front-end is linear: z_lr = X_s @ coef.T + intercept
# Gradient of BCE loss w.r.t. standardised input X_s:
#   dL/dX_s ≈ (p_mlp - y) * lr.coef_.flatten()
# This is exact for a pure LR; for LR→MLP it is a first-order
# approximation via the LR pre-layer (as stated in Section 3.8
# of the paper: "gradients from the trained LR–MLP (typically
# the LR pre-layer)").

def _predict(pipe, mlp, X_std):
    """Predict probabilities for standardised inputs."""
    z = pipe.lr.decision_function(X_std).reshape(-1, 1)
    return mlp.predict_proba(z)[:, 1]

def _lr_gradient(pipe, mlp, X_std, y):
    """
    Signed gradient of BCE w.r.t. standardised input.
    Returns sign_grad shape (n, n_features).
    """
    p    = _predict(pipe, mlp, X_std)           # (n,)
    dloss = (p - y.astype(float))               # dL/dp ≈ p - y for BCE
    coef  = pipe.lr.coef_.flatten()             # (n_features,)
    grad  = np.outer(dloss, coef)               # (n, n_features)
    return np.sign(grad)

def fgsm_sklearn(X_std, y, pipe, mlp, eps, clip_min, clip_max):
    """
    Single-step FGSM in standardised feature space.
    Perturbations clipped to [clip_min, clip_max] per feature.
    """
    sign_grad = _lr_gradient(pipe, mlp, X_std, y)
    X_adv = X_std + eps * sign_grad
    return np.clip(X_adv, clip_min, clip_max)

def pgd_sklearn(X_std, y, pipe, mlp, eps, alpha, steps, clip_min, clip_max):
    """
    PGD (multi-step FGSM) in standardised feature space.
    Projects back to l-inf ball of radius eps after each step.
    """
    X_adv = X_std.copy()
    for _ in range(steps):
        sign_grad = _lr_gradient(pipe, mlp, X_adv, y)
        X_adv = X_adv + alpha * sign_grad
        # Project onto l-inf ball centred at original X_std
        X_adv = np.clip(X_adv, X_std - eps, X_std + eps)
        X_adv = np.clip(X_adv, clip_min, clip_max)
    return X_adv

def constrained_pgd_sklearn(X_std, y, pipe, mlp, eps, alpha,
                             steps, clip_min, clip_max, mask):
    """
    Constrained PGD (Eq. 7 in paper) — restricts gradient updates
    to attacker-actionable features only via binary mask M.

    mask : np.ndarray of shape (n_features,), dtype float
           M_i = 1.0 if feature i is actionable (attacker can modify),
           M_i = 0.0 if non-actionable (protocol fixed, addressing, flags).

    All other behaviour identical to pgd_sklearn():
      - l-inf ball projection after each step
      - per-feature training-range clipping
    """
    X_adv = X_std.copy()
    for _ in range(steps):
        sign_grad = _lr_gradient(pipe, mlp, X_adv, y)
        sign_grad = sign_grad * mask          # ← Eq. 7: zero non-actionable gradients
        X_adv = X_adv + alpha * sign_grad
        X_adv = np.clip(X_adv, X_std - eps, X_std + eps)
        X_adv = np.clip(X_adv, clip_min, clip_max)
    return X_adv


def compute_clip_bounds(X_std_train):
    """Per-feature min/max from training set (standardised space)."""
    return X_std_train.min(axis=0), X_std_train.max(axis=0)

print("[OK] FGSM + PGD (sklearn LR-gradient) loaded")

[OK] FGSM + PGD (sklearn LR-gradient) loaded


In [16]:
# ============================================================
# W3 + W4 DIAGNOSTICS (updated — includes constrained PGD)
# Q1: FN count under fixed clean threshold
# Q2: Score distribution shift
# + tau95 ceiling effect, CI overlap, constrained PGD
# ============================================================

def w3_diagnostics(pipe, mlp, X_train_std, X_test_std, y_test,
                   dataset_name, outdir, seed=42,
                   eps_list=[0.05, 0.10], pgd_steps=10,
                   actionable_mask=None):
    """
    Run W3+W4 diagnostics for one trained model on one test set.

    actionable_mask : np.ndarray shape (n_features,) or None
        Binary mask M_i=1.0 if feature i is attacker-actionable.
        If None, constrained PGD is skipped (unconstrained only).
        Build with build_actionable_mask(feature_names) in Cell 10.
    """
    ensure_outdir(outdir)
    clip_min, clip_max = compute_clip_bounds(X_train_std)

    # --- Clean baseline ---
    p_clean   = _predict(pipe, mlp, X_test_std)
    fpr_clean, tau_clean = fpr_at_dr(y_test, p_clean, target_dr=0.95)
    auroc_clean = roc_auc_score(y_test, p_clean)

    attack_mask_idx = (y_test == 1)
    benign_mask_idx = (y_test == 0)
    n_actionable = int(actionable_mask.sum()) if actionable_mask is not None else None

    print(f"\n{'='*65}")
    print(f"W3+W4 DIAGNOSTICS — {dataset_name} (seed={seed})")
    if actionable_mask is not None:
        print(f"  Actionable features: {n_actionable}/{len(actionable_mask)} "
              f"({n_actionable/len(actionable_mask)*100:.1f}%)")
    print(f"{'='*65}")
    print(f"Clean: AUROC={auroc_clean:.6f}  FPR@95%DR={fpr_clean:.2e}  "
          f"tau95={tau_clean:.8f}")
    print(f"  Attack score: mean={p_clean[attack_mask_idx].mean():.6f}  "
          f"min={p_clean[attack_mask_idx].min():.6f}  "
          f"max={p_clean[attack_mask_idx].max():.6f}")
    print(f"  Benign score: mean={p_clean[benign_mask_idx].mean():.6f}  "
          f"min={p_clean[benign_mask_idx].min():.6f}  "
          f"max={p_clean[benign_mask_idx].max():.6f}")

    summary = {
        "dataset": dataset_name, "seed": seed,
        "n_actionable_features": n_actionable,
        "n_total_features": X_test_std.shape[1],
        "clean": {
            "auroc": float(auroc_clean),
            "fpr95": float(fpr_clean) if fpr_clean == fpr_clean else None,
            "tau95": float(tau_clean) if tau_clean == tau_clean else None,
            "attack_score_mean": float(p_clean[attack_mask_idx].mean()),
            "benign_score_mean": float(p_clean[benign_mask_idx].mean()),
        },
        "adversarial": []
    }

    for eps in eps_list:
        alpha = CFG["adv"]["pgd_alpha_factor"] * eps

        # Build attack list: always FGSM + PGD; add cPGD if mask provided
        attack_list = [
            ("FGSM", fgsm_sklearn(X_test_std, y_test, pipe, mlp,
                                   eps, clip_min, clip_max)),
            ("PGD",  pgd_sklearn (X_test_std, y_test, pipe, mlp,
                                   eps, alpha, pgd_steps, clip_min, clip_max)),
        ]
        if actionable_mask is not None:
            attack_list.append((
                "cPGD",
                constrained_pgd_sklearn(X_test_std, y_test, pipe, mlp,
                                        eps, alpha, pgd_steps,
                                        clip_min, clip_max,
                                        actionable_mask)
            ))

        for attack_name, X_adv in attack_list:
            p_adv = _predict(pipe, mlp, X_adv)
            fpr_adv,  tau_adv  = fpr_at_dr(y_test, p_adv, target_dr=0.95)
            auroc_adv = roc_auc_score(y_test, p_adv)

            atk_shift    = p_adv[attack_mask_idx].mean() - p_clean[attack_mask_idx].mean()
            benign_shift = p_adv[benign_mask_idx].mean() - p_clean[benign_mask_idx].mean()

            if tau_clean == tau_clean:
                y_hat_fixed = (p_adv >= tau_clean).astype(int)
                tn_f, fp_f, fn_f, tp_f = confusion_matrix(
                    y_test, y_hat_fixed, labels=[0,1]).ravel()
                fn_rate_fixed = fn_f / attack_mask_idx.sum() if attack_mask_idx.sum()>0 else np.nan
                fpr_fixed     = fp_f / (fp_f + tn_f + 1e-12)
            else:
                fn_f, fn_rate_fixed, fpr_fixed = np.nan, np.nan, np.nan

            tau_ceiling = (tau_adv >= 1.0 - 1e-8) if tau_adv == tau_adv else False

            label = f"[{attack_name} eps={eps}]"
            if attack_name == "cPGD":
                label += f" [constrained: {n_actionable}/{len(actionable_mask)} features]"

            print(f"\n  {label}")
            print(f"    AUROC={auroc_adv:.6f}  FPR@95%DR={fpr_adv:.2e}  "
                  f"tau95={tau_adv:.8f}  CEILING={tau_ceiling}")
            print(f"    Attack score shift : {atk_shift:+.6f}  "
                  f"({p_clean[attack_mask_idx].mean():.6f} → "
                  f"{p_adv[attack_mask_idx].mean():.6f})")
            print(f"    Benign score shift : {benign_shift:+.6f}  "
                  f"({p_clean[benign_mask_idx].mean():.6f} → "
                  f"{p_adv[benign_mask_idx].mean():.6f})")
            if tau_clean == tau_clean:
                print(f"    Fixed-threshold FN count: {fn_f}  "
                      f"(FN rate={fn_rate_fixed:.4f})  FPR={fpr_fixed:.2e}")
                fpr_diff = fpr_adv - fpr_clean if (fpr_adv==fpr_adv and fpr_clean==fpr_clean) else np.nan
                if fpr_diff == fpr_diff:
                    direction = "DECREASE" if fpr_diff < 0 else "INCREASE"
                    print(f"    FPR change vs clean: {fpr_diff:+.2e}  → {direction}")
                    if tau_ceiling:
                        print(f"    *** tau95 CEILING EFFECT ***")

            summary["adversarial"].append({
                "attack": attack_name, "eps": eps,
                "constrained": (attack_name == "cPGD"),
                "auroc": float(auroc_adv),
                "fpr95_retuned": float(fpr_adv) if fpr_adv==fpr_adv else None,
                "tau95": float(tau_adv) if tau_adv==tau_adv else None,
                "tau95_ceiling": bool(tau_ceiling),
                "attack_score_mean_adv": float(p_adv[attack_mask_idx].mean()),
                "attack_score_shift": float(atk_shift),
                "benign_score_mean_adv": float(p_adv[benign_mask_idx].mean()),
                "benign_score_shift": float(benign_shift),
                "fn_count_fixed_threshold": int(fn_f) if fn_f==fn_f else None,
                "fn_rate_fixed_threshold": float(fn_rate_fixed) if fn_rate_fixed==fn_rate_fixed else None,
                "fpr_fixed_threshold": float(fpr_fixed) if fpr_fixed==fpr_fixed else None,
            })

    save_json(summary, os.path.join(outdir,
              f"w3w4_diagnostics_{dataset_name}_seed{seed}.json"))
    return summary

print("[OK] w3_diagnostics (WITH constrained PGD / W4) loaded")

[OK] w3_diagnostics (WITH constrained PGD / W4) loaded


In [17]:
# ============================================================
# CI Overlap test (Table 6 CIC-IoMT FPR improvement)
# ============================================================

def ci_overlap_test(fpr_clean_mean, fpr_clean_std,
                    fpr_adv_mean,   fpr_adv_std,
                    label=""):
    """
    Test whether the FPR 'improvement' under attack is statistically
    significant by checking 95% CI overlap (seed-averaged values).
    """
    z = 1.96
    clean_lo = fpr_clean_mean - z * fpr_clean_std
    clean_hi = fpr_clean_mean + z * fpr_clean_std
    adv_lo   = fpr_adv_mean   - z * fpr_adv_std
    adv_hi   = fpr_adv_mean   + z * fpr_adv_std

    overlap  = not (clean_lo > adv_hi or adv_lo > clean_hi)
    diff     = fpr_adv_mean - fpr_clean_mean

    print(f"\n{'='*60}")
    print(f"CI Overlap Test: {label}")
    print(f"  Clean FPR: {fpr_clean_mean:.2e} ± {fpr_clean_std:.2e}")
    print(f"    95% CI : [{clean_lo:.2e}, {clean_hi:.2e}]")
    print(f"  Adv FPR : {fpr_adv_mean:.2e}  ± {fpr_adv_std:.2e}")
    print(f"    95% CI : [{adv_lo:.2e},  {adv_hi:.2e}]")
    print(f"  Mean difference : {diff:+.2e}")
    print(f"  CIs OVERLAP     : {overlap}")
    if overlap:
        print(f"  => NOT statistically significant. FPR 'improvement'")
        print(f"     is within measurement noise across seeds.")
    else:
        print(f"  => Statistically significant at 95% level.")
    return overlap

# --- Reproduce Table 6 CI test from paper ---
print("Reproducing Table 6 CI overlap test (seed-averaged values from paper):")

# CIC-IoMT: clean vs FGSM eps=0.05
ci_overlap_test(
    fpr_clean_mean=3.72e-4, fpr_clean_std=2.82e-4,
    fpr_adv_mean  =2.51e-4, fpr_adv_std  =1.06e-4,
    label="CIC-IoMT: Clean vs FGSM eps=0.05"
)
ci_overlap_test(
    fpr_clean_mean=3.72e-4, fpr_clean_std=2.82e-4,
    fpr_adv_mean  =2.08e-4, fpr_adv_std  =9.77e-5,
    label="CIC-IoMT: Clean vs PGD  eps=0.05"
)
ci_overlap_test(
    fpr_clean_mean=3.72e-4, fpr_clean_std=2.82e-4,
    fpr_adv_mean  =2.29e-4, fpr_adv_std  =9.77e-5,
    label="CIC-IoMT: Clean vs PGD  eps=0.10"
)

Reproducing Table 6 CI overlap test (seed-averaged values from paper):

CI Overlap Test: CIC-IoMT: Clean vs FGSM eps=0.05
  Clean FPR: 3.72e-04 ± 2.82e-04
    95% CI : [-1.81e-04, 9.25e-04]
  Adv FPR : 2.51e-04  ± 1.06e-04
    95% CI : [4.32e-05,  4.59e-04]
  Mean difference : -1.21e-04
  CIs OVERLAP     : True
  => NOT statistically significant. FPR 'improvement'
     is within measurement noise across seeds.

CI Overlap Test: CIC-IoMT: Clean vs PGD  eps=0.05
  Clean FPR: 3.72e-04 ± 2.82e-04
    95% CI : [-1.81e-04, 9.25e-04]
  Adv FPR : 2.08e-04  ± 9.77e-05
    95% CI : [1.65e-05,  3.99e-04]
  Mean difference : -1.64e-04
  CIs OVERLAP     : True
  => NOT statistically significant. FPR 'improvement'
     is within measurement noise across seeds.

CI Overlap Test: CIC-IoMT: Clean vs PGD  eps=0.10
  Clean FPR: 3.72e-04 ± 2.82e-04
    95% CI : [-1.81e-04, 9.25e-04]
  Adv FPR : 2.29e-04  ± 9.77e-05
    95% CI : [3.75e-05,  4.20e-04]
  Mean difference : -1.43e-04
  CIs OVERLAP     : True
 

True

In [18]:
# ============================================================
# Seed-averaged adversarial evaluation (W3 + W4)
# Reproduces Table 6 + adds cPGD rows
# ============================================================

def run_adv_seed_avg(dataset_name, split_fn, outdir,
                     use_smote=True, seeds=None,
                     actionable_mask=None):
    """
    Train LR→MLP over multiple seeds, run W3+W4 diagnostics,
    aggregate seed-averaged mean ± std.

    actionable_mask : output of build_actionable_mask(), or None.
    """
    if seeds is None:
        seeds = SEEDS

    all_rows = []
    for seed in seeds:
        X_train, y_train, X_test, y_test = split_fn(seed)

        pipe, mlp = fit_lr_then_mlp(
            X_train, y_train, X_test, y_test,
            use_smote=use_smote, seed=seed,
            max_epochs=CFG["train"]["max_epochs"],
            batch_size=CFG["train"]["batch_size"],
            hidden_units=CFG["train"]["mlp_hidden_units"],
        )
        X_train_s = pipe.scaler.transform(X_train)
        X_test_s  = pipe.scaler.transform(X_test)

        diag = w3_diagnostics(
            pipe, mlp, X_train_s, X_test_s, y_test,
            dataset_name, outdir, seed=seed,
            eps_list=CFG["adv"]["eps_list"],
            pgd_steps=CFG["adv"]["pgd_steps"],
            actionable_mask=actionable_mask,
        )
        all_rows.append(diag)

    # ---- Seed-averaged table ----
    from collections import defaultdict
    adv_by_key = defaultdict(list)
    for r in all_rows:
        for entry in r["adversarial"]:
            key = (entry["attack"], entry["eps"])
            adv_by_key[key].append(entry)

    print(f"\n{'='*75}")
    print(f"SEED-AVERAGED TABLE — {dataset_name}")
    print(f"{'='*75}")
    hdr = f"{'Attack':8s} {'eps':5s} | {'AUROC':>16s} {'FPR@95%DR':>18s} {'tau95':>16s} | {'Atk shift':>10s}"
    print(hdr)
    print("-"*90)

    # Clean row
    clean_aurocs = [r["clean"]["auroc"] for r in all_rows]
    clean_fprs   = [r["clean"]["fpr95"] for r in all_rows if r["clean"]["fpr95"] is not None]
    clean_taus   = [r["clean"]["tau95"] for r in all_rows if r["clean"]["tau95"] is not None]
    print(f"{'Clean':8s} {'0.00':5s} | "
          f"{np.mean(clean_aurocs):.4f}±{np.std(clean_aurocs):.4f}   "
          f"{np.mean(clean_fprs):.2e}±{np.std(clean_fprs):.2e}   "
          f"{np.mean(clean_taus):.6f}±{np.std(clean_taus):.6f} | —")

    # Determine attack order: FGSM, PGD, then cPGD (if present)
    attack_names = ["FGSM", "PGD"]
    if actionable_mask is not None:
        attack_names.append("cPGD")

    for attack in attack_names:
        for eps in CFG["adv"]["eps_list"]:
            rows_k = adv_by_key.get((attack, eps), [])
            if not rows_k:
                continue
            aurocs    = [x["auroc"] for x in rows_k]
            fprs      = [x["fpr95_retuned"] for x in rows_k if x["fpr95_retuned"] is not None]
            taus      = [x["tau95"] for x in rows_k if x["tau95"] is not None]
            atk_shift = [x["attack_score_shift"] for x in rows_k]
            ceiling   = any(x["tau95_ceiling"] for x in rows_k)
            tag = " [constrained]" if attack == "cPGD" else ""
            print(f"{attack+tag:22s} {eps:5.2f} | "
                  f"{np.mean(aurocs):.4f}±{np.std(aurocs):.4f}   "
                  f"{np.mean(fprs):.2e}±{np.std(fprs):.2e}   "
                  f"{np.mean(taus):.6f}±{np.std(taus):.6f} | "
                  f"{np.mean(atk_shift):+.4f}")

    return all_rows

print("[OK] run_adv_seed_avg (WITH constrained PGD / W4) loaded")

[OK] run_adv_seed_avg (WITH constrained PGD / W4) loaded


In [19]:
# ============================================================
# W4: Actionable mask builder  (definition only — no data needed)
# Run this cell before Cell 11.
# ============================================================

# Actionable feature keywords (Section 5.2 / Eq. 7 of paper)
ACTIONABLE_KEYWORDS = [
    "packet_rate", "pkt_rate", "pkts/s", "packets/s",
    "byte_rate",   "bytes/s",  "bwd_packet_rate", "fwd_packet_rate",
    "bwd_pkt",     "fwd_pkt",  "bwd_byte", "fwd_byte",
    "mean_packet", "mean_pkt", "avg_packet", "avg_pkt",
    "total_fwd",   "total_bwd", "tot_fwd",   "tot_bwd",
    "pkt_size",    "packet_size", "seg_size",
    "flow_duration", "duration", "flow_iat", "mean_iat",
    "iat_mean",    "iat_std",   "iat_max",   "iat_min",
    "idle",        "active",
]

NON_ACTIONABLE_KEYWORDS = [
    "is_tcp", "is_udp", "protocol", "proto",
    "src_port", "dst_port", "sport", "dport",
    "fin_flag", "syn_flag", "rst_flag", "psh_flag",
    "ack_flag", "urg_flag", "cwe_flag", "ece_flag",
    "flag_count", "flag_cnt",
]

def build_actionable_mask(feature_names, verbose=True):
    """
    Build binary mask M of shape (n_features,) for constrained PGD (Eq. 7).
    M_i = 1.0  →  feature i is attacker-actionable (gradients allowed)
    M_i = 0.0  →  non-actionable (protocol/address/flag field, zeroed out)

    Strategy: default all to actionable; explicitly zero non-actionable keywords.
    """
    n    = len(feature_names)
    mask = np.ones(n, dtype=np.float32)

    actionable_hits    = []
    non_actionable_hits = []
    default_hits        = []

    for i, name in enumerate(feature_names):
        nl = name.lower().replace(" ", "_").replace("-", "_")
        is_non = any(kw in nl for kw in NON_ACTIONABLE_KEYWORDS)
        is_act = any(kw in nl for kw in ACTIONABLE_KEYWORDS)

        if is_non:
            mask[i] = 0.0
            non_actionable_hits.append((i, name))
        elif is_act:
            mask[i] = 1.0
            actionable_hits.append((i, name))
        else:
            mask[i] = 1.0   # conservative default
            default_hits.append((i, name))

    if verbose:
        print(f"  Total features     : {n}")
        print(f"  Actionable (kw)    : {len(actionable_hits)}")
        print(f"  Non-actionable     : {len(non_actionable_hits)}")
        print(f"  Default-actionable : {len(default_hits)}")
        print(f"  M=1 total          : {int(mask.sum())}/{n} "
              f"({mask.mean()*100:.1f}%)")
        if non_actionable_hits:
            print("  Non-actionable features set to M=0:")
            for idx, nm in non_actionable_hits:
                print(f"    [{idx:03d}] {nm}")
        if default_hits:
            print(f"  Default-actionable (review if needed):")
            for idx, nm in default_hits[:10]:
                print(f"    [{idx:03d}] {nm}")
            if len(default_hits) > 10:
                print(f"    ... and {len(default_hits)-10} more")
    return mask

print("[OK] build_actionable_mask defined (ACTIONABLE/NON_ACTIONABLE keywords loaded)")
print("     Run Cell 11 (entry point) next.")

[OK] build_actionable_mask defined (ACTIONABLE/NON_ACTIONABLE keywords loaded)
     Run Cell 11 (entry point) next.


In [21]:
# ============================================================
# Entry point — full W3 + W4 evaluation
# Requires: Cells 00-10 already executed.
# ============================================================

import pandas as pd

# ---- 1. Load datasets ----
nf_df     = pd.read_csv(CFG["paths"]["nf"],        low_memory=False)
cic_tr_df = pd.read_csv(CFG["paths"]["cic_train"], low_memory=False)
cic_te_df = pd.read_csv(CFG["paths"]["cic_test"],  low_memory=False)

# ---- 2. Build feature lists ----
nf_feats     = build_feature_list(nf_df, [
    CFG["columns"]["nf_label_binary"],
    CFG["columns"]["nf_label_mc"], "Attack"])

cic_feats_tr = build_feature_list(cic_tr_df, [
    CFG["columns"]["cic_label_binary"],
    CFG["columns"]["cic_label_mc"]])

print(f"[INFO] NF  features : {len(nf_feats)}")
print(f"[INFO] CIC features : {len(cic_feats_tr)}")

# ---- 3. Print full feature lists (for your verification) ----
print("\n=== NF-ToN-IoT features ===")
for i, f in enumerate(nf_feats):
    print(f"  [{i:03d}] {f}")

print(f"\n=== CIC-IoMT features ===")
for i, f in enumerate(cic_feats_tr):
    print(f"  [{i:03d}] {f}")

# ---- 4. Build actionable masks (W4 / Eq. 7) ----
print("\n=== Building actionable masks ===")
print("NF-ToN-IoT mask:")
nf_mask  = build_actionable_mask(nf_feats,    verbose=True)
print("\nCIC-IoMT mask:")
cic_mask = build_actionable_mask(cic_feats_tr, verbose=True)
print("\n[Review the mask assignments above.]")
print("[If any feature is miscategorised, update ACTIONABLE_KEYWORDS")
print(" or NON_ACTIONABLE_KEYWORDS in Cell 10 and re-run Cells 10-11.]")

# ---- 5. Split functions ----
def make_nf_split_fn(df, feats, bin_col):
    X = df[feats].values
    y = df[bin_col].values.astype(int)
    def fn(seed):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                random_state=seed, stratify=y)
        return X_tr, y_tr, X_te, y_te
    return fn

def make_cic_split_fn(df_tr, df_te, feats):
    df_all = pd.concat([df_tr, df_te], ignore_index=True)
    bin_col = CFG["columns"]["cic_label_binary"]
    y_all   = build_binary_labels(df_all, bin_col, None)
    common  = [f for f in feats if f in df_all.columns]
    X_all   = df_all[common].values
    def fn(seed):
        X_tr, X_tmp, y_tr, y_tmp = train_test_split(
            X_all, y_all, test_size=0.40,
            random_state=seed, stratify=y_all)
        X_val, X_te, y_val, y_te = train_test_split(
            X_tmp, y_tmp, test_size=0.50,
            random_state=seed, stratify=y_tmp)
        return X_tr, y_tr, X_te, y_te
    return fn

nf_split  = make_nf_split_fn(nf_df, nf_feats,
                               CFG["columns"]["nf_label_binary"])
cic_split = make_cic_split_fn(cic_tr_df, cic_te_df, cic_feats_tr)

# ---- 6. Run evaluations ----
print("\n[Running NF-ToN-IoT ...]")
nf_results = run_adv_seed_avg(
    "NF-ToN-IoT", nf_split,
    outdir="./outputs/w3w4",
    use_smote=True,
    actionable_mask=nf_mask,
)

print("\n[Running CIC-IoMT-OptionA ...]")
cic_results = run_adv_seed_avg(
    "CIC-IoMT-OptionA", cic_split,
    outdir="./outputs/w3w4",
    use_smote=True,
    actionable_mask=cic_mask,
)

print("\n[DONE] All results saved to ./outputs/w3w4/")

[INFO] NF  features : 10
[INFO] CIC features : 45

=== NF-ToN-IoT features ===
  [000] L4_SRC_PORT
  [001] L4_DST_PORT
  [002] PROTOCOL
  [003] L7_PROTO
  [004] IN_BYTES
  [005] OUT_BYTES
  [006] IN_PKTS
  [007] OUT_PKTS
  [008] TCP_FLAGS
  [009] FLOW_DURATION_MILLISECONDS

=== CIC-IoMT features ===
  [000] Header_Length
  [001] Protocol Type
  [002] Duration
  [003] Rate
  [004] Srate
  [005] Drate
  [006] fin_flag_number
  [007] syn_flag_number
  [008] rst_flag_number
  [009] psh_flag_number
  [010] ack_flag_number
  [011] ece_flag_number
  [012] cwr_flag_number
  [013] ack_count
  [014] syn_count
  [015] fin_count
  [016] rst_count
  [017] HTTP
  [018] HTTPS
  [019] DNS
  [020] Telnet
  [021] SMTP
  [022] SSH
  [023] IRC
  [024] TCP
  [025] UDP
  [026] DHCP
  [027] ARP
  [028] ICMP
  [029] IGMP
  [030] IPv
  [031] LLC
  [032] Tot sum
  [033] Min
  [034] Max
  [035] AVG
  [036] Std
  [037] Tot size
  [038] IAT
  [039] Number
  [040] Magnitue
  [041] Radius
  [042] Covariance
  [043] 